# PYMAGSIMS

## Analysing mass spectra obtained by FPD

In [ ]:
from pymagsims.spectrum import Spectrum
from pymagsims import spectrum
from pymagsims.isotopes import load_builtin_isotopes
from pymagsims.isotopes import load_builtin_isotopes

%load_ext autoreload
%autoreload 2

In [ ]:
spec = Spectrum.from_main_analysis_file("../data/FPD_01_2604281458290.csv")
spec.plot(log_y=True)

In [ ]:
from pymagsims.spectrum import Spectrum

main_file = "../data/FPD_01_2604281458290.csv"
raw_file = "../data/FPD_01_2604281458290.raw"

spec = Spectrum.from_main_analysis_file(main_file)

raw_spec = Spectrum.from_raw_file(
    raw_file,
    calibration=spec,
    min_channel=1,
    max_channel=12000,
)


In [ ]:
spec.plot(log_y=True)

In [ ]:
raw_spec.plot(log_y=True)

In [ ]:
peaks = spec.find_peaks(
    prominence=200,
    distance=5,
)

peaks.tail(30)

In [ ]:
isotopes = load_builtin_isotopes()

assignments = spec.assign_peaks(
    isotope_table=isotopes,
    tolerance=0.05,
    prominence=200,
    distance=5,
)

assignments.head(30)

In [ ]:
fig, ax, assignments = spec.plot_with_peaks(
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=100,
    distance=5,
    log_y=False,
    xlim=(30,80)
)

#assignments.head(20)

In [ ]:
spec.plot_with_element_markers(
    isotope_table=isotopes,
    elements=["Cu", "Zn", "Ga"],
    log_y=True,
    min_abundance=0.1,
    xlim=(50,80)
)

In [ ]:
from pymagsims.interactive import plot_spectrum_interactive

fig = plot_spectrum_interactive(
    spec,
    log_y=True,
)

fig.show()

In [ ]:
from pymagsims.isotopes import load_builtin_isotopes

isotopes = load_builtin_isotopes()

fig, assignments = spec.plot_with_peaks_interactive(
    isotope_table=isotopes,
    prominence=100,
    tolerance=0.2,
    xlim=(20, 40),
    log_y=True,
)

fig

In [ ]:
fig = spec.plot_with_element_markers_interactive(
    isotope_table=isotopes,
    elements=["Cu", "Ga", "Zn"],
    xlim=(10, 80),
    log_y=True,
    min_abundance=0.1,
)

fig

In [ ]:
from pymagsims.interactive import manual_peak_binner_interactive

manual_bins = manual_peak_binner_interactive(spec, log_y=True, xlim=(20, 40))

In [ ]:
bins = manual_bins.to_dataframe()
bins

In [ ]:
assignments = spec.assign_peaks(
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=100,
    distance=5,
)

bins = spec.create_bins_from_assignments(
    assignments,
    width=0.3,
)

bins

In [ ]:
fig = spec.plot_interactive(log_y=True, xlim=(bins["mass_min"].min() - 1, bins["mass_max"].max() + 1))

for _, b in bins.iterrows():
    fig.add_vrect(
        x0=b["mass_min"],
        x1=b["mass_max"],
        fillcolor="LightSalmon",
        opacity=0.3,
        line_width=0,
        annotation_text=b["label"],
        annotation_position="top left",
    )

fig

## FPD image analysis

### Raw data analysis

In [ ]:
from pymagsims.raw_image import SIMSRawImage

raw_img = SIMSRawImage.from_fpd_raw("../data/FPD_image2.raw")

raw_img.metadata

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tic = raw_img.total_ion_image()

plt.figure(figsize=(6, 6))
plt.imshow(np.log1p(tic), cmap="viridis")
plt.colorbar(label="log(1 + counts)")
plt.title("Total ion image")
plt.show()

In [ ]:
img = raw_img.image_from_channel_bin(6960, 6990)

plt.figure(figsize=(6, 6))
plt.imshow(np.log1p(img), cmap="viridis")
plt.colorbar(label="log(1 + counts)")
plt.title("Channel bin 6960–6990")
plt.show()

In [ ]:
from pymagsims import spectrum
from pymagsims.raw_image import SIMSRawImage

spec = Spectrum.from_main_analysis_file("../data/FPD_01_2604281458290.csv")
raw_img = SIMSRawImage.from_fpd_raw("../data/FPD_image2.raw")

In [ ]:
img_28 = raw_img.image_from_mass_bin(
    spectrum=spec,
    mass_min=60,
    mass_max=66,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(6, 6))
plt.imshow(np.log1p(img_28), cmap="viridis")
plt.colorbar(label="log(1 + counts)")
plt.title("Mass bin 27.8–28.2 amu")
plt.show()

In [ ]:
from pymagsims.isotopes import load_builtin_isotopes

isotopes = load_builtin_isotopes()

assignments = spec.assign_peaks(
    isotope_table=isotopes,
    tolerance=0.3,
    prominence=100,
    distance=2,
)

bins = spec.create_bins_from_assignments(
    assignments,
    width=0.3,
)

bins

In [ ]:
images = raw_img.images_from_bins(
    spectrum=spec,
    bins=bins,
)

In [ ]:
images.keys()

In [ ]:
label = list(images.keys())[5]

plt.figure(figsize=(6, 6))
plt.imshow(np.log1p(images[label]), cmap="viridis")
plt.colorbar(label="log(1 + counts)")
plt.title(label)
plt.show()

In [ ]:
from pymagsims.plotting import plot_ion_image

plot_ion_image(img_28, title="Mass 28 amu")

In [ ]:
from pymagsims.plotting import plot_ion_image_grid

fig, axes = plot_ion_image_grid(
    images,
    log=True,
    ncols=3,
)

In [ ]:
images = raw_img.images_from_bins(
    spectrum=spec,
    bins=bins,
    include_total=True,
)

plot_ion_image_grid(images, log=True, ncols=3);

In [ ]:
#fig, axes = plot_ion_image_grid(
#    images,
#    log=True,
#    ncols=3,
#    cmaps=["viridis", "magma", "cividis", "turbo"],
#)

In [ ]:
from pymagsims.image import SIMSImage

img = SIMSImage.from_fpd_csv("../data/FPD_image2.csv")

img.metadata

In [ ]:
img.roi_table

In [ ]:
img.spectrum.data.head()

In [ ]:
img.labels()

In [ ]:
img.total().shape

In [ ]:
from pymagsims.plotting import plot_ion_image

plot_ion_image(
    img.total(),
    log=True,
    title="Total ion image from FPD CSV",
)

In [ ]:
from pymagsims.plotting import plot_ion_image_grid

plot_ion_image_grid(
    img.images,
    log=True,
    ncols=2,
)

In [ ]:
## Depth profile

In [ ]:
from pymagsims import SIMSDepthProfile

depth = SIMSDepthProfile.from_fpd_csv(
    "../data/Depthprofile2.csv"
)

depth.profiles.head()

In [ ]:
depth.plot(log_y=True)

In [ ]:
from pymagsims import SIMSDepthProfile

depth_raw = SIMSDepthProfile.from_fpd_raw(
    "../data/Depthprofile2.raw",
    roi_table=depth.roi_table,
    skip_initial=16,
)

depth_raw.profiles.head()

In [ ]:
depth_raw.plot(log_y=True)

In [ ]:
depth.profiles["All channels"].reset_index(drop=True).equals(
    depth_raw.profiles["All channels"].reset_index(drop=True)
)

## 3D images

In [ ]:
from pymagsims import SIMSDepthProfile

depth_csv = SIMSDepthProfile.from_fpd_csv(
    "../data/3dimage.csv"
)

depth_raw = SIMSDepthProfile.from_fpd_raw(
    "../data/3dimage_Histogram.raw",
    roi_table=depth_csv.roi_table,
    skip_initial=16,
)

#merged = depth_raw.merged_spectrum()
#merged.plot(x="Channel", y="Amplitude", log_y=True)

merged = depth_raw.merged_spectrum(
    calibration=depth_csv.last_spectrum
)

In [ ]:
from pathlib import Path
from pymagsims import Spectrum, SIMSVolume
from pymagsims.isotopes import load_builtin_isotopes

spec = Spectrum.from_main_analysis_file("../data/FPD_01_2604281458290.csv")
isotopes = load_builtin_isotopes()

assignments = spec.assign_peaks(
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=100,
    distance=5,
)

bins = spec.create_bins_from_assignments(assignments, width=0.3)

paths = sorted(Path("../data/3d").glob("*Image_*.raw"))

volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    spectrum=spec,
    bins=bins.head(5),
    include_total=True,
    shape=(256, 256),
)

volume.labels()

In [ ]:
from pymagsims.plotting import plot_volume_slice, plot_ion_image

plot_volume_slice(volume, label="Total", z=5, log=True)

In [ ]:
depth_raw.raw_histograms

In [ ]:
from pymagsims.isotopes import load_builtin_isotopes

isotopes = load_builtin_isotopes()

#merged = depth_raw.merged_spectrum(calibration=depth.last_spectrum)

merged.plot(log_y=True)

assignments_merged = merged.assign_peaks(
    isotope_table=isotopes,
    tolerance=100,
    prominence=1,
    distance=1,
)

assignments_merged

In [ ]:
layer_0 = depth_raw.spectrum_for_layer(
    layer=0,
    calibration=merged,
)

layer_0.plot(log_y=True)

assignments_layer_0 = layer_0.assign_peaks(
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=50,
    distance=5,
)

assignments_layer_0.head(30)

In [ ]:
layer_assignments = {}

for i in range(len(depth_raw.raw_histograms)):
    layer_spec = depth_raw.spectrum_for_layer(
        layer=i,
        calibration=merged,
    )

    layer_assignments[i] = layer_spec.assign_peaks(
        isotope_table=isotopes,
        tolerance=0.2,
        prominence=50,
        distance=5,
    )

In [ ]:
bins = merged.create_bins_from_assignments(
    assignments_merged,
    width=0.3,
)



In [ ]:
depth_elements = SIMSDepthProfile.from_fpd_raw(
    "../data/Depthprofile2.raw",
    bins=bins,
    spectrum=merged,
    skip_initial=16,
)

depth_elements.plot(log_y=True)

In [ ]:
volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    spectrum=depth_csv.last_spectrum,
    bins=bins.head(5),
    include_total=True,
    shape=(256, 256),
)

volume.bins

In [ ]:
volume.depth_profile("14N")


In [ ]:
plot_volume_slice(volume, label="14N", z=0, log=True)